# Vestibular-BR — treino SFT + GRPO (Unsloth)

GPU T4.

In [ ]:
GITHUB_REPO = "https://github.com/MariaEduarda-jg/Text-to-SQL.git"
HF_USER = "MariaEduarda-jg"

import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for k in ["HF_TOKEN", "GEMINI_API_KEY"]:
    try:
        os.environ[k] = secrets.get_secret(k)
    except Exception:
        pass

In [ ]:
!git clone -q $GITHUB_REPO repo
%cd repo
!pip install -q unsloth
!pip install -q -e .
!vbr data

In [ ]:
!wc -l data/sft/train_sft.jsonl

## SFT

In [ ]:
!python -m vestibular.train.sft --out outputs/sft --push-to {HF_USER}/vestibular-br-2b-sft

## GRPO

In [ ]:
import glob
diff = glob.glob("results/runs/train__vllm-Qwen-Qwen3.5-2B__caption__pass4/predictions.jsonl")
diff_arg = f"--difficulty-from {diff[0]}" if diff else ""
!python -m vestibular.train.grpo --out outputs/grpo --max-steps 250 {diff_arg} --push-to {HF_USER}/vestibular-br-2b-grpo

## SFT + GRPO

In [ ]:
!python -m vestibular.train.grpo --init-adapter outputs/sft --out outputs/sft-grpo --max-steps 250 {diff_arg} --push-to {HF_USER}/vestibular-br-2b-sft-grpo

In [ ]:
!mkdir -p results/training/grpo results/training/sft-grpo
!cp outputs/grpo/log_history.json results/training/grpo/ ; cp outputs/sft-grpo/log_history.json results/training/sft-grpo/
!zip -qr /kaggle/working/treino.zip results/training